# 🚀 Fine-Tune Venture-Coder (Qwen 14B) on Luau with Unsloth

This notebook fine-tunes `Qwen2.5-Coder-14B-Instruct` using **QLoRA** on a free Google Colab **T4 GPU (16 GB VRAM)**.

### What this accomplishes:
1. Teaches the model the **Roblox Senior Engineering Codex** (--!strict, Services pattern, session locking, Selene linting).
2. Trains on captured repair attempts (`data/training/repairs.jsonl`) so the model stops dropping tokens and syntax.
3. Exports directly to **GGUF format** for 1-click import into **Ollama** on your local machine.

## 1. Install Unsloth & Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" "trl<0.9.0" peft accelerate bitsandbytes datasets

## 2. Load Base Model (Qwen2.5-Coder-14B 4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
dtype = None  # None for auto-detection (Float16 on T4)
load_in_4bit = True  # Fits 14B comfortably inside 16GB VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 3. Upload and Prepare Dataset (`repairs.jsonl`)

In [ ]:
from google.colab import files
import os

print("Upload your repairs.jsonl file from data/training/repairs.jsonl:")
uploaded = files.upload()
dataset_path = list(uploaded.keys())[0]
print(f"Loaded {dataset_path}")

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = load_dataset("json", data_files=dataset_path, split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

## 4. Train Model with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

trainer_stats = trainer.train()

## 5. Export directly to GGUF (for Ollama)

In [ ]:
# Export directly to quantized GGUF format
model.save_pretrained_gguf("venture_coder_model", tokenizer, quantization_method="q4_k_m")

# Download the GGUF file to your PC
from google.colab import files
import glob

gguf_files = glob.glob("venture_coder_model/*.gguf")
if gguf_files:
    print(f"Downloading {gguf_files[0]}...")
    files.download(gguf_files[0])
else:
    print("Model saved to venture_coder_model directory.")

## 6. How to Import into Local Ollama

Once the `.gguf` file downloads to your local PC, create a `Modelfile` in the same directory:
```text
FROM ./venture_coder_model-q4_k_m.gguf
PARAMETER temperature 0.1
PARAMETER top_p 0.95
```
Then run in your PowerShell terminal:
```powershell
ollama create venture-coder:14b -f Modelfile
```
Your fine-tuned model is now active and ready for the Roblox Engineer Agent!